In [1]:
import duckdb
from pathlib import Path
import polars as pl 
import sys

# Adds the parent directory of the notebook (i.e., project root) to sys.path
sys.path.append(str(Path().resolve().parent))

import src.transform_utils as tu


In [2]:
db_file = Path("../data/NSQIP_A_Clean.duckdb")

print(f"Using DuckDB database: {db_file}")

if db_file.exists():
    size_mb = db_file.stat().st_size / (1024 * 1024 * 1024)
    print(f"File exists. Size: {size_mb:.2f} GB")
else:
    print("File does not exist.")
    
# get table name (only one table in the database)
with duckdb.connect(db_file) as con:
    tables = con.execute("SHOW TABLES").fetchall()
    print(f"Tables in the database: {tables}")
    table_name = tables[0][0] if tables else None
    print(f"Table name: {table_name}")

Using DuckDB database: ..\data\NSQIP_A_Clean.duckdb
File exists. Size: 4.22 GB
Tables in the database: [('all_data_table',)]
Table name: all_data_table


In [4]:
# cols that can be changed to categorical
to_categorical = ['ADMQTR', 'ADMSYR', 'ADMYR', 'CASEID',
                  'CONCPT8', 'CONCPT9', 'HDISDT', 'OPERYR',
                  'PGY', 'PUFYEAR', 'READMUNRELICD93',
                  'READMUNRELICD94', 'READMUNRELICD95',
                  'YRDEATH']

In [7]:
tu.cast_column_in_place(
    db_path=db_file,
    table_name=table_name,
    columns_to_cast=to_categorical,
)

Column 'ADMQTR' casted to STRING in table 'all_data_table'.
Column 'ADMSYR' casted to STRING in table 'all_data_table'.
Column 'ADMYR' casted to STRING in table 'all_data_table'.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Column 'CASEID' casted to STRING in table 'all_data_table'.
Column 'CONCPT8' casted to STRING in table 'all_data_table'.
Column 'CONCPT9' casted to STRING in table 'all_data_table'.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Column 'HDISDT' casted to STRING in table 'all_data_table'.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Column 'OPERYR' casted to STRING in table 'all_data_table'.
Column 'PGY' casted to STRING in table 'all_data_table'.
Column 'PUFYEAR' casted to STRING in table 'all_data_table'.
Column 'READMUNRELICD93' casted to STRING in table 'all_data_table'.
Column 'READMUNRELICD94' casted to STRING in table 'all_data_table'.
Column 'READMUNRELICD95' casted to STRING in table 'all_data_table'.
Column 'YRDEATH' casted to STRING in table 'all_data_table'.
All columns casted to STRING in table 'all_data_table'.


In [4]:
# needs adjustment: 'AGE'
# 90+ causing issues, make AGE__AS_INT, and AGE_IS_90_PLUS 

tu.fix_age_column(
    db_path=db_file,
    table_name=table_name,
)

In [5]:
# columns to make CPT list from
cpt_cols = ['CPT', 'CONCPT1', 'CONCPT2', 'CONCPT3', 'CONCPT4', 
            'CONCPT5', 'CONCPT6', 'CONCPT7', 'CONCPT8', 'CONCPT9',
            'CONCPT10', 'OTHERCPT1', 'OTHERCPT2', 'OTHERCPT3',
            'OTHERCPT4', 'OTHERCPT5', 'OTHERCPT6', 'OTHERCPT7',
            'OTHERCPT8', 'OTHERCPT9', 'OTHERCPT10']

In [6]:
tu.add_combined_cpt_column(
    db_path=db_file,
    table_name=table_name,
    cpt_columns=cpt_cols,
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
# Sum of RVU cols for initial surgery
rvu_cols = ['CONWRVU1', 'CONWRVU2', 'CONWRVU3', 'CONWRVU4',
            'CONWRVU5', 'CONWRVU6', 'CONWRVU7', 'CONWRVU8',
            'CONWRVU9', 'CONWRVU10', 'OTHERWRVU1', 'OTHERWRVU2',
            'OTHERWRVU3', 'OTHERWRVU4', 'OTHERWRVU5', 'OTHERWRVU6',
            'OTHERWRVU7', 'OTHERWRVU8', 'OTHERWRVU9', 'OTHERWRVU10',
            'WORKRVU']

In [10]:
tu.add_total_rvu(
    db_path=db_file,
    table_name=table_name,
    rvu_columns=rvu_cols,
)

In [3]:
# MORTPROB is string becuase of scientific notation
# need to convert to float

tu.cast_string_column_to_numeric(
    db_path=db_file,
    table_name=table_name,
    column_name='MORTPROB',
)

Successfully casted column 'MORTPROB' to DOUBLE


In [13]:
# RACE ends 2007, RACE_NEW starts 2008, combine into RACE

tu.combine_race_cols(
    db_path=db_file,
    table_name=table_name,
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
# Columns with multiple values separated by commas
separate_cols = ['ANESTHES_OTHER', 'IMMUNO_CAT', 'OP_APPROACH']

In [ ]:
tu.split_comma_separated_columns(
    db_path=db_file,
    table_name=table_name,
    column_names=separate_cols,
)

Dropped existing column 'ANESTHES_OTHER_list'.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created list column 'ANESTHES_OTHER_list' from 'ANESTHES_OTHER'.
Dropped existing column 'IMMUNO_CAT_list'.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created list column 'IMMUNO_CAT_list' from 'IMMUNO_CAT'.
Dropped existing column 'OP_APPROACH_list'.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
"""
15756	Free muscle flap
15757	Free skin flap
15758	Free fascial flap
15842	Free muscle for facial paralysis
20962	Free bone flap other than fibula, iliac crest, or metatarsal
20969	Free osteocutaneous flap other than iliac crest, metatarsal, or great toe
20955	Free bone flap
43496	Free jejunum transfer
"""
stop

free_flap_cpt = [ '15756', '15757', '15758', '20969', '43496', '20955', 
                 '20962', '15842', '20956', '20957', '20970']

soft_tissue_flap = ['15756', '15757', '15758', '43496', '15842']

bone_flap = ['20970', '20957', '20956', '20955', '20962', '20969']

In [ ]:
stop

tu.add_free_flap_indicator(
    db_path=db_file,
    table_name=table_name,
)

